# GPS + IMU Sensor Fusion for Trajectory Estimation
## Theory Notebook — Everything You Need to Understand Before Writing a Single Line of Code

**Author:** Prem M Patel | Computer Engineering | Sankalchand Patel University  
**Contact:** prempatel7740@gmail.com

---

> This notebook is purely theory. No code. Just concepts, equations, and intuition.  
> Read this first. Then open the implementation notebook and everything will make sense.

---

## What Problem Are We Solving?

You are walking through a shopping mall with your phone. Your phone has two sensors — GPS and IMU. Neither of them alone can tell you exactly where you are at all times.

**GPS** knows your location on Earth, but it updates slowly (once per second) and loses signal the moment you step indoors. It also jumps around by several meters due to signal noise.

**IMU** (accelerometer + gyroscope) feels every tiny movement you make, 100 times per second. But it has a problem — small errors in each reading add up over time and the position estimate drifts further and further from reality.

The question this project answers is: **can we combine these two imperfect sensors to get one accurate, continuous position estimate — even when GPS dies completely?**

Yes. And the answer involves a 60-year-old algorithm called the Kalman Filter, and a deep learning model called LSTM. The same combination that guides rockets and satellites.

---

## 1. The Sensors — What They Measure and Why They're Imperfect

### 1.1 Accelerometer

An accelerometer measures **linear acceleration** — how fast your velocity is changing — along three axes (X, Y, Z). The unit is metres per second squared (m/s²).

If you hold your phone still, the accelerometer still reads about 9.8 m/s² along one axis. That is gravity. The phone cannot tell the difference between gravitational pull and physical acceleration, which is one of the fundamental challenges in inertial navigation.

When you walk, each footstep creates a small spike in the accelerometer signal. That spike is real motion, but it sits on top of high-frequency vibration noise that has nothing to do with your actual displacement.

**Key equation — dead reckoning with acceleration:**

$$v(t) = v(t_0) + \int_{t_0}^{t} a(\tau) \, d\tau$$

$$p(t) = p(t_0) + \int_{t_0}^{t} v(\tau) \, d\tau$$

Where:
- $p(t)$ = position at time $t$
- $v(t)$ = velocity at time $t$  
- $a(t)$ = measured acceleration

The problem: every tiny measurement error in $a(t)$ gets integrated once into velocity error, then integrated again into position error. After 60 seconds, a 0.01 m/s² bias in the accelerometer produces a position drift of **18 metres**. This is why IMU alone fails for long-term navigation.

---

### 1.2 Gyroscope

A gyroscope measures **angular velocity** — how fast you are rotating — around three axes. The unit is radians per second (rad/s).

When the phone is held steady, gyroscope readings should be near zero. When you turn a corner or twist your wrist, the gyroscope spikes. It tells us the **heading** — which direction you are facing — which is essential for converting body-frame acceleration into world-frame displacement.

Gyroscopes also drift. At 100 Hz over several minutes, even a tiny constant bias accumulates into a significant heading error. This is called **gyroscope drift** and it is one of the central problems in inertial navigation.

---

### 1.3 GPS

GPS (Global Positioning System) works by receiving signals from at least 4 satellites and computing your position through trilateration. It gives you latitude, longitude, and altitude directly — no integration required.

**GPS strengths:**
- Absolute position — no drift over time
- Works anywhere outdoors with clear sky view

**GPS weaknesses:**
- Updates at ~1 Hz only (one reading per second)
- Accuracy of 3–5 metres under normal conditions
- Loses signal indoors, in tunnels, underground
- Affected by multipath errors (signal bouncing off buildings)
- Completely unreliable above 30 km altitude (relevant for ISRO)

**GPS coordinate system:**

GPS gives coordinates in degrees of latitude and longitude. For local navigation calculations, we convert to a flat Cartesian system using the flat-earth approximation:

$$x = R \cdot \Delta\lambda \cdot \cos(\phi_0)$$

$$y = R \cdot \Delta\phi$$

Where:
- $R = 6{,}371{,}000$ m = Earth radius
- $\Delta\phi = \phi - \phi_0$ = latitude difference from origin (radians)
- $\Delta\lambda = \lambda - \lambda_0$ = longitude difference from origin (radians)
- $\phi_0$ = reference latitude

This approximation holds well for distances under ~50 km.

---

## 2. Signal Processing — Cleaning the Sensor Data

### 2.1 Why Raw IMU Data is Noisy

When you record 100 accelerometer readings per second from a walking person, you get two types of signal mixed together:

1. **Useful signal** — the actual motion: walking pace, direction changes, stops
2. **Noise** — high-frequency vibrations from footsteps, phone casing resonance, electronic interference

The noise sits at high frequencies (above ~5–10 Hz for a walking person). The real motion signal sits at low frequencies (below ~5 Hz). We want to keep the low frequencies and throw away the high frequencies.

This is exactly what a **low-pass filter** does.

---

### 2.2 The Butterworth Low-Pass Filter

A Butterworth filter is designed to have a maximally flat frequency response in the passband — meaning it does not distort the frequencies it keeps. It is widely used in signal processing for exactly this kind of sensor data cleaning.

**Transfer function in the frequency domain:**

$$|H(j\omega)|^2 = \frac{1}{1 + \left(\frac{\omega}{\omega_c}\right)^{2n}}$$

Where:
- $\omega$ = signal frequency
- $\omega_c$ = cutoff frequency (we use 5 Hz)
- $n$ = filter order (we use 4)

In plain English: frequencies below 5 Hz pass through mostly unchanged. Frequencies above 5 Hz get progressively attenuated. Order 4 means the rolloff is steep — a clean cut.

**The Nyquist theorem** tells us the maximum frequency we can represent when sampling at rate $f_s$:

$$f_{Nyquist} = \frac{f_s}{2}$$

At 100 Hz sampling rate, the Nyquist frequency is 50 Hz. Our cutoff of 5 Hz is normalised as $5/50 = 0.1$ before passing to the filter design function.

**Why filtfilt instead of just filter?**

Applying a filter in one direction introduces a **phase delay** — the output signal is shifted in time relative to the input. For navigation, a time-shifted signal would place you at the wrong position at the wrong time.

`filtfilt` applies the filter **twice** — once forward in time, once backward. The two phase shifts cancel exactly, giving **zero phase distortion**. This is essential for accurate timestamp-aligned data.

---

## 3. Timestamp Synchronisation — Getting Sensors on the Same Clock

### 3.1 The Sampling Rate Problem

IMU runs at 100 Hz — 100 readings per second, one every 10 milliseconds.  
GPS runs at 1 Hz — 1 reading per second, one every 1000 milliseconds.

This means for every 1 GPS reading, there are 100 IMU readings. You cannot directly combine them because they exist at different points in time.

| Time (s) | IMU reading | GPS reading |
|---|---|---|
| 0.000 | Yes | Yes |
| 0.010 | Yes | No |
| 0.020 | Yes | No |
| ... | Yes | No |
| 1.000 | Yes | Yes |

We need a GPS value at every IMU timestamp. The solution is **linear interpolation**.

---

### 3.2 Linear Interpolation

Given two known GPS readings at times $t_1$ and $t_2$, with positions $p_1$ and $p_2$, we estimate the position at any time $t$ between them as:

$$p(t) = p_1 + \frac{t - t_1}{t_2 - t_1} \cdot (p_2 - p_1)$$

This assumes the person moved at a **constant velocity** between GPS readings. Since GPS updates every second and humans walk at roughly constant speed on short timescales, this is a reasonable approximation.

After interpolation, every IMU timestamp has a corresponding GPS position, and we can build a unified synchronised dataset.

---

## 4. The Kalman Filter — The Heart of This Project

### 4.1 What It Is and Where It Came From

Rudolf Kálmán published his filter in 1960. Within two years it was being used in the Apollo program to navigate spacecraft to the Moon. Today it is in every commercial aircraft, every modern car with stability control, every smartphone with navigation, and every rocket ever launched by ISRO.

It is not an exaggeration to say the Kalman Filter is one of the most important algorithms ever written.

The core idea is elegant: **maintain a probability distribution over possible states, and update it optimally every time you receive a new measurement.**

---

### 4.2 State Space Representation

We model the navigation system using a **state vector** $\mathbf{x}$ that contains everything we want to know:

$$\mathbf{x} = \begin{bmatrix} p_x \\ p_y \\ v_x \\ v_y \end{bmatrix}$$

Where:
- $p_x, p_y$ = position in metres (East and North from starting point)
- $v_x, v_y$ = velocity in m/s

The filter estimates this state at every timestep. We never observe it directly — we observe noisy GPS measurements and noisy IMU readings. The filter figures out the most likely true state from these imperfect observations.

---

### 4.3 The State Transition Matrix F

Between timesteps, the state evolves according to physics. If you are at position $p_x$ moving at velocity $v_x$, then after time $\Delta t$:

$$p_x^{new} = p_x + v_x \cdot \Delta t$$

Written as a matrix multiplication $\mathbf{x}_{k} = \mathbf{F} \mathbf{x}_{k-1}$:

$$\mathbf{F} = \begin{bmatrix} 1 & 0 & \Delta t & 0 \\ 0 & 1 & 0 & \Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

With $\Delta t = 0.01$ s (100 Hz IMU). This is a **constant velocity motion model** — we assume velocity is constant between IMU updates (a good approximation at 100 Hz).

---

### 4.4 The Measurement Matrix H

GPS can only observe position, not velocity. So our measurement $\mathbf{z}$ is:

$$\mathbf{z} = \begin{bmatrix} p_x^{GPS} \\ p_y^{GPS} \end{bmatrix}$$

The measurement matrix $\mathbf{H}$ maps the full state to what we can actually measure:

$$\mathbf{H} = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix}$$

This says: from the state vector, pick out position only. Ignore velocity — we cannot observe that directly.

---

### 4.5 The Noise Matrices

**Process noise Q** — how much we trust our IMU-based prediction:

$$\mathbf{Q} = \sigma_q^2 \mathbf{I}_4$$

A small $\mathbf{Q}$ means we trust the IMU model heavily. A large $\mathbf{Q}$ means we expect the state to vary unpredictably. We use $\sigma_q^2 = 0.1$.

**Measurement noise R** — how much we trust GPS:

$$\mathbf{R} = \sigma_r^2 \mathbf{I}_2$$

GPS has ~5 metre accuracy, so $\sigma_r^2 = 5.0$. This tells the filter: GPS readings are uncertain by about 5 metres.

---

### 4.6 The Two-Step Algorithm

The Kalman Filter runs two steps at every timestep:

**Step 1 — Predict** (using IMU):

$$\hat{\mathbf{x}}_{k|k-1} = \mathbf{F} \mathbf{x}_{k-1}$$

$$\mathbf{P}_{k|k-1} = \mathbf{F} \mathbf{P}_{k-1} \mathbf{F}^T + \mathbf{Q}$$

The state is propagated forward using physics. The uncertainty $\mathbf{P}$ grows — because we are predicting, not measuring.

**Step 2 — Update** (using GPS):

$$\mathbf{K}_k = \mathbf{P}_{k|k-1} \mathbf{H}^T \left( \mathbf{H} \mathbf{P}_{k|k-1} \mathbf{H}^T + \mathbf{R} \right)^{-1}$$

$$\mathbf{x}_k = \hat{\mathbf{x}}_{k|k-1} + \mathbf{K}_k \left( \mathbf{z}_k - \mathbf{H} \hat{\mathbf{x}}_{k|k-1} \right)$$

$$\mathbf{P}_k = \left( \mathbf{I} - \mathbf{K}_k \mathbf{H} \right) \mathbf{P}_{k|k-1}$$

$\mathbf{K}_k$ is the **Kalman Gain** — the key quantity. It decides how much to trust the GPS correction vs the prediction.

- If GPS is very accurate ($\mathbf{R}$ small): $\mathbf{K}$ is large → trust GPS heavily
- If IMU prediction is very accurate ($\mathbf{P}$ small): $\mathbf{K}$ is small → trust prediction

The innovation $\left( \mathbf{z}_k - \mathbf{H} \hat{\mathbf{x}}_{k|k-1} \right)$ is the difference between what GPS measured and what we predicted. The filter corrects the state by a fraction of this innovation, controlled by $\mathbf{K}$.

After the update, uncertainty $\mathbf{P}$ shrinks — because we just received real data.

---

### 4.7 Why This Works When GPS Dies

When GPS is lost, we simply skip the update step and only run the predict step. The IMU keeps integrating acceleration into velocity into position. The uncertainty $\mathbf{P}$ grows with each step (because we are predicting without correction), but the position estimate continues moving forward.

This is called **dead reckoning** — navigation purely from inertial measurements. It works well for short periods (5–30 seconds). Over longer periods, IMU drift becomes significant. That is exactly the problem the LSTM model is designed to help solve.

---

## 5. LSTM Neural Networks — Teaching AI to Navigate

### 5.1 Why a Standard Neural Network Is Not Enough

A plain feedforward neural network takes a fixed input and produces a fixed output. It has no memory — each prediction is independent of the previous ones.

IMU data is fundamentally sequential. The pattern of the last 2 seconds of sensor readings tells you far more about current position than any single reading in isolation. To use that temporal context, we need a network with memory.

---

### 5.2 Recurrent Neural Networks and the Vanishing Gradient Problem

A Recurrent Neural Network (RNN) solves this by feeding the hidden state from timestep $t-1$ into the computation at timestep $t$:

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

But during backpropagation through time, gradients are multiplied together at every step. Over long sequences, gradients either **vanish** (become near zero — earlier timesteps learn nothing) or **explode** (become enormous — training becomes unstable).

For a 200-step sequence like ours, vanilla RNNs simply do not work.

---

### 5.3 LSTM — Long Short-Term Memory

Hochreiter and Schmidhuber introduced LSTM in 1997 specifically to solve the vanishing gradient problem. The key innovation is the **cell state** $C_t$ — a separate memory that runs through the entire sequence with only minor linear interactions, allowing gradients to flow freely across hundreds of timesteps.

The LSTM cell has three gates that control information flow:

**Forget Gate** — what to throw away from memory:

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

**Input Gate** — what new information to store:

$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$

$$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$$

**Cell State Update:**

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

**Output Gate** — what to output as hidden state:

$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$

$$h_t = o_t \odot \tanh(C_t)$$

Where:
- $\sigma$ = sigmoid function: $\sigma(x) = \frac{1}{1 + e^{-x}}$ (outputs 0 to 1)
- $\tanh$ = hyperbolic tangent (outputs -1 to 1)
- $\odot$ = element-wise multiplication
- $W_f, W_i, W_C, W_o$ = learned weight matrices

In plain English: the LSTM learns **what to remember, what to forget, and what to output** at each step. After training, it has learned patterns like "this particular combination of accelerometer spikes followed by gyroscope values means the person is turning left at this speed."

---

### 5.4 Sliding Window — How We Feed Data to the LSTM

Our sensor data is one long sequence of 25,971 timesteps. The LSTM needs fixed-size inputs. We use a **sliding window** approach:

```
Full sequence:  [t0, t1, t2, t3, t4, t5, t6, t7, t8 ...]

Window 1:  [t0  → t199]  → predict position at t199
Window 2:  [t50 → t249]  → predict position at t249
Window 3:  [t100 → t299] → predict position at t299
```

- Window size: 200 timesteps = 2 seconds of IMU data
- Step size: 50 timesteps = 0.5 seconds
- 75% overlap between consecutive windows
- More overlap = more training samples

Each window contains a `(200, 6)` array — 200 timesteps of 6 sensor channels (acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z). The target is the 2D position at the last timestep.

---

### 5.5 Our Model Architecture

```
Input: (200 timesteps, 6 sensor channels)
         ↓
LSTM(128 units, return_sequences=True)
  — reads all 200 steps, outputs at every step
  — learns short-term patterns
         ↓
BatchNormalization + Dropout(0.3)
         ↓
LSTM(64 units, return_sequences=False)
  — reads all 200 steps, outputs only final summary
  — learns long-range patterns across the window
         ↓
BatchNormalization + Dropout(0.3)
         ↓
Dense(32, activation='relu')
  — non-linear combination of LSTM features
         ↓
Dropout(0.2)
         ↓
Dense(2)
  — output: predicted fused_x, fused_y
```

**Total parameters: ~121,000** — all learned during training.

---

### 5.6 Loss Function and Optimiser

**Loss: Mean Squared Error (MSE)**

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \left( \hat{y}_i - y_i \right)^2$$

MSE penalises large errors heavily (because of the square). A prediction 10 metres off is penalised 100× more than a prediction 1 metre off. This is appropriate for navigation — large position errors are much worse than small ones.

**Optimiser: Adam**

Adam (Adaptive Moment Estimation) adapts the learning rate for each parameter individually based on first and second moment estimates of the gradients:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$

$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

$$\theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

Default values: $\alpha = 0.001$, $\beta_1 = 0.9$, $\beta_2 = 0.999$. Adam is the standard first-choice optimiser for most deep learning tasks.

---

### 5.7 Overfitting and How We Prevent It

Overfitting happens when the model memorises the training data instead of learning general patterns. It performs well on training data but fails on new data.

We use three techniques:

**Dropout** — randomly set 30% of neurons to zero during each training step. Forces the network to learn redundant representations. Neurons cannot rely on specific neighbours, so they must individually carry useful information.

**BatchNormalization** — normalises activations within each mini-batch. Reduces internal covariate shift, allowing higher learning rates and more stable training. Has a slight regularising effect.

**EarlyStopping** — monitor validation loss. If it stops improving for 8 consecutive epochs, stop training and restore the best weights seen so far. Prevents training past the optimal point.

**ReduceLROnPlateau** — if validation loss plateaus for 4 epochs, halve the learning rate. Allows fine-grained convergence near the optimum.

---

## 6. Evaluation Metrics — How We Measure Success

### 6.1 Mean Absolute Error (MAE)

$$\text{MAE} = \frac{1}{N} \sum_{i=1}^{N} |\hat{y}_i - y_i|$$

Simple average of absolute errors. Easy to interpret — if MAE = 0.5 m, predictions are off by 0.5 m on average. Not sensitive to outliers.

### 6.2 Root Mean Square Error (RMSE)

$$\text{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2}$$

Squares each error before averaging, then takes the square root. More sensitive to large errors than MAE. Better metric for navigation — a 10 m error once is much worse than ten 1 m errors.

### 6.3 Euclidean Distance Error

For 2D position prediction, the actual positional error at each sample is the straight-line distance between predicted and true position:

$$e_i = \sqrt{(\hat{x}_i - x_i)^2 + (\hat{y}_i - y_i)^2}$$

This is the most meaningful metric for this project — it tells you directly how many metres away from the true position the LSTM predicted.

---

## 7. The ADVIO Dataset

ADVIO (Apple Device Visual Inertial Odometry) was collected by Aalto University in Finland. 23 sequences of a person walking through real environments — shopping malls, metro stations, offices, streets — recorded on a standard iPhone.

| Property | Value |
|---|---|
| Sequences | 23 (advio-01 to advio-23) |
| Environments | Mall, metro, office, street |
| IMU rate | ~100 Hz |
| GPS rate | ~1 Hz |
| Total size | ~2.3 GB |

**Files in each sequence:**

| File | Contents |
|---|---|
| `accelerometer.csv` | timestamp, acc_x, acc_y, acc_z (m/s²) |
| `gyro.csv` | timestamp, gyro_x, gyro_y, gyro_z (rad/s) |
| `platform-locations.csv` | timestamp, lat, lon, alt, accuracy |
| `ground-truth/pose.csv` | high-accuracy reference position + orientation |

The ground truth was captured using a separate high-accuracy tracking system. We use the Kalman-fused position (not raw ground truth pose) as our LSTM training target, since it represents the practically achievable best estimate from the available sensors.

---

## 8. Why This Matters for ISRO / IISU

ISRO's Inertial Systems Unit (IISU) in Thiruvananthapuram designs and manufactures the inertial navigation systems used in every ISRO launch vehicle — PSLV, GSLV, LVM3, and the upcoming Gaganyaan human spaceflight mission.

The physics is identical to what we implemented. Only the scale and precision requirements are different.

| This Project | ISRO Rocket |
|---|---|
| Person walking in a mall | Rocket flying through atmosphere |
| Smartphone accelerometer | IISU-built ring laser gyroscope + accelerometer |
| GPS loses signal indoors | GPS unreliable above 30 km altitude |
| Kalman Filter fusing GPS + IMU | Core navigation algorithm in every ISRO vehicle |
| LSTM predicting during GPS loss | Onboard dead reckoning during telemetry blackout |
| Ground truth poses.csv | Actual rocket trajectory from range safety radar |

The Kalman Filter equations you implemented — the F matrix, H matrix, predict step, update step — are in the flight software of every rocket that has ever been launched from Sriharikota.

---

## 9. Summary — What You Need Going Into the Code

Before opening the implementation notebook, make sure you understand these five ideas:

1. **Why both sensors are needed** — GPS drifts long-term and loses signal; IMU drifts short-term but is always available. Together they cover each other's weaknesses.

2. **What the Kalman Filter does** — it maintains a probability distribution over the state (position + velocity) and updates it optimally using physics (predict step) and measurements (update step). The Kalman Gain $\mathbf{K}$ decides how much to trust each source.

3. **Why we filter IMU data** — high-frequency vibration noise contaminates the signal. A Butterworth low-pass filter removes noise above 5 Hz while preserving real motion below 5 Hz.

4. **Why we use a sliding window for LSTM** — the LSTM needs fixed-size inputs. A 200-step window gives the model 2 seconds of history to predict position from. More history = better context.

5. **What the LSTM actually learns** — it learns the mapping from IMU sequences to position changes. After training, it can predict where you are using only accelerometer and gyroscope data, with no GPS at all.

---

*You are now ready for the implementation notebook. Every line of code directly implements one of the concepts above.*